# WIMBD Arabic — Method Testing Notebook
Each cell tests one method with a sample input and prints the output clearly.

## Setup
Import the module. Make sure `wimbd_arabic.py` is in the same folder.

In [26]:
import wimbd_arabic as W
print("Module loaded successfully")
print(f"Stopwords loaded: {len(W.ARABIC_STOPWORDS):,} words")

Module loaded successfully
Stopwords loaded: 13,285 words


## 1. `normalize_ar(text)`
Strips diacritics (tashkeel), removes tatweel, and unifies alef/ya/ta_marbuta variants.

In [27]:
# Test: remove tashkeel (diacritics)
text = "مُحَمَّدٌ يَذْهَبُ إِلى المَدْرَسَةِ"
result = W.normalize_ar(text)
print(f"IN : {text}")
print(f"OUT: {result}")

IN : مُحَمَّدٌ يَذْهَبُ إِلى المَدْرَسَةِ
OUT: محمد يذهب الي المدرسه


In [28]:
# Test: unify alef variants (أ إ آ → ا)
for text in ["أحمد", "إسلام", "آمين"]:
    result = W.normalize_ar(text)
    print(f"IN : {text}  →  OUT: {result}")

IN : أحمد  →  OUT: احمد
IN : إسلام  →  OUT: اسلام
IN : آمين  →  OUT: امين


In [29]:
# Test: unify ya (ى → ي) and ta marbuta (ة → ه)
for text in ["مبنى", "مدرسة"]:
    result = W.normalize_ar(text)
    print(f"IN : {text}  →  OUT: {result}")

IN : مبنى  →  OUT: مبني
IN : مدرسة  →  OUT: مدرسه


In [30]:
# Test: remove tatweel (elongation)
text = "جمـــــيل"
result = W.normalize_ar(text)
print(f"IN : {text}  →  OUT: {result}")

IN : جمـــــيل  →  OUT: جميل


In [31]:
# Full sentence combining all cases
text = "أنـــــا سعيدٌ جداً بوجودي في هذه الوَرْشَةِ"
result = W.normalize_ar(text)
print(f"IN : {text}")
print(f"OUT: {result}")

IN : أنـــــا سعيدٌ جداً بوجودي في هذه الوَرْشَةِ
OUT: انا سعيد جدا بوجودي في هذه الورشه


## 2. `tokenize(text)`
Splits text into a list of tokens (Arabic words, Latin words, and numbers). Punctuation is excluded.

In [32]:
text = "الذكاء الاصطناعي يعتمد على التعلم الآلي"
tokens = W.tokenize(text)
print(f"IN : {text}")
print(f"OUT: {tokens}")

IN : الذكاء الاصطناعي يعتمد على التعلم الآلي
OUT: ['الذكاء', 'الاصطناعي', 'يعتمد', 'على', 'التعلم', 'الآلي']


In [33]:
# Mixed Arabic and Latin
text = "تعلم Python و JavaScript في عام 2024"
tokens = W.tokenize(text)
print(f"IN : {text}")
print(f"OUT: {tokens}")

IN : تعلم Python و JavaScript في عام 2024
OUT: ['تعلم', 'Python', 'و', 'JavaScript', 'في', 'عام', '2024']


In [34]:
# Punctuation is excluded
text = "مرحبا، كيف حالك؟ أنا بخير!"
tokens = W.tokenize(text)
print(f"IN : {text}")
print(f"OUT: {tokens}")

IN : مرحبا، كيف حالك؟ أنا بخير!
OUT: ['مرحبا', 'كيف', 'حالك', 'أنا', 'بخير']


In [35]:
import wimbd_arabic as W
import collections

# Full pipeline: normalize → tokenize → remove stopwords
# ARABIC_STOPWORDS is now normalized with normalize_ar() so
# إن/أن → ان and إلى → الي are correctly filtered out
text = "أنـــا سعيدٌ جداً بوجودي في هذه الوَرْشَةِ مع United Nations عام 2022"
normalized = W.normalize_ar(text)
tokens = W.tokenize(normalized)
tokens_nostop = [t for t in tokens if t not in W.ARABIC_STOPWORDS and len(t) > 1]

print(f"IN         : {text}")
print(f"normalized : {normalized}")
print(f"tokens     : {tokens}")
print(f"no stops   : {tokens_nostop}")
print()
print("Stopword normalization fix applied:")
print("  ARABIC_STOPWORDS = {normalize_ar(w) for w in _ar_sw.stopwords_list()}")
print("  إن/أن → ان, إلى → الي, هن → هن, والى handled correctly now.")

# Verify هن and والى are in the normalized stopwords list
for w in ['ان', 'الي', 'هن', 'وال']:
    print(f"  '{w}' in ARABIC_STOPWORDS: {w in W.ARABIC_STOPWORDS}")


IN         : أنـــا سعيدٌ جداً بوجودي في هذه الوَرْشَةِ مع United Nations عام 2022
normalized : انا سعيد جدا بوجودي في هذه الورشه مع United Nations عام 2022
tokens     : ['انا', 'سعيد', 'جدا', 'بوجودي', 'في', 'هذه', 'الورشه', 'مع', 'United', 'Nations', 'عام', '2022']
no stops   : ['سعيد', 'بوجودي', 'الورشه', 'United', 'Nations', 'عام', '2022']

Stopword normalization fix applied:
  ARABIC_STOPWORDS = {normalize_ar(w) for w in _ar_sw.stopwords_list()}
  إن/أن → ان, إلى → الي, هن → هن, والى handled correctly now.
  'ان' in ARABIC_STOPWORDS: True
  'الي' in ARABIC_STOPWORDS: True
  'هن' in ARABIC_STOPWORDS: True
  'وال' in ARABIC_STOPWORDS: False


## 3. `fingerprint(text)`
Returns an MD5 hash of the text. Used for exact duplicate detection — identical texts produce the same hash.

In [36]:
text = "الذكاء الاصطناعي"
fp = W.fingerprint(text)
print(f"IN : {text!r}")
print(f"OUT: {fp}  (length: {len(fp)})")

IN : 'الذكاء الاصطناعي'
OUT: 972ded6d9d8efc6f71aa701a85048d86  (length: 32)


In [37]:
# Same text → same fingerprint
text = "الذكاء الاصطناعي"
fp1 = W.fingerprint(text)
fp2 = W.fingerprint(text)
print(f"First call : {fp1}")
print(f"Second call: {fp2}")
print(f"Same? {fp1 == fp2}")

First call : 972ded6d9d8efc6f71aa701a85048d86
Second call: 972ded6d9d8efc6f71aa701a85048d86
Same? True


In [38]:
# Different texts → different fingerprints
texts = ["الذكاء الاصطناعي", "تعلم الآلة", "معالجة اللغة الطبيعية"]
for t in texts:
    print(f"{t!r:30s}  →  {W.fingerprint(t)}")

'الذكاء الاصطناعي'              →  972ded6d9d8efc6f71aa701a85048d86
'تعلم الآلة'                    →  f71c17d32526b0bf0d767880237c6607
'معالجة اللغة الطبيعية'         →  4b80c19090c58929568bd1378e4e3a1b


## 4. `PII_PATTERNS`
Regex patterns for detecting personally identifiable information. Categories: email, url, ipv4, phone, card_like.

In [39]:
# Email detection
text = "تواصل معنا على البريد الإلكتروني ahmed@example.com أو info@kds.ae"
matches = W.PII_PATTERNS["email"].findall(text)
print(f"IN    : {text}")
print(f"emails: {matches}")

IN    : تواصل معنا على البريد الإلكتروني ahmed@example.com أو info@kds.ae
emails: ['ahmed@example.com', 'info@kds.ae']


In [40]:
# IPv4 detection
text = "عنوان الخادم هو 192.168.1.100 ويمكن الوصول عبر 10.0.0.1"
matches = W.PII_PATTERNS["ipv4"].findall(text)
print(f"IN   : {text}")
print(f"ipv4 : {matches}")

IN   : عنوان الخادم هو 192.168.1.100 ويمكن الوصول عبر 10.0.0.1
ipv4 : ['192.168.1.100', '10.0.0.1']


In [41]:
# Phone number detection
text = "اتصل بنا على 0501234567 أو +966-555-1234"
matches = W.PII_PATTERNS["phone"].findall(text)
print(f"IN    : {text}")
print(f"phones: {matches}")

IN    : اتصل بنا على 0501234567 أو +966-555-1234
phones: ['0501234567', '+966-555-1234']


In [42]:
# All PII categories on one sentence
text = "راسلنا على user@example.com أو اتصل 0501234567 والخادم على 192.168.1.1"
print(f"IN: {text}\n")
for kind, pat in W.PII_PATTERNS.items():
    matches = pat.findall(text)
    print(f"  {kind:<12}: {matches}")

IN: راسلنا على user@example.com أو اتصل 0501234567 والخادم على 192.168.1.1

  email       : ['user@example.com']
  url         : []
  ipv4        : ['192.168.1.1']
  phone       : ['0501234567']
  card_like   : []


## 5. Offensive content detection
Checks if any token in the text matches the Arabic offensive word blocklist (after normalization).

In [43]:
def check_offensive(text):
    tokens = set(W.tokenize(W.normalize_ar(text)))
    hits = tokens & W.OFFENSIVE_WORDS
    print(f"IN  : {text}")
    print(f"HITS: {hits if hits else 'none'}")
    print()

check_offensive("هذا الرجل مثل كلب")
check_offensive("الذكاء الاصطناعي مفيد للغاية")
check_offensive("هذا كَلْبٌ")          # with diacritics — still caught after normalization
check_offensive("كلب حمار غبي")        # multiple hits
check_offensive("وكلب")               # waw prefix — NOT caught (known limitation)

IN  : هذا الرجل مثل كلب
HITS: {'كلب'}

IN  : الذكاء الاصطناعي مفيد للغاية
HITS: none

IN  : هذا كَلْبٌ
HITS: {'كلب'}

IN  : كلب حمار غبي
HITS: {'كلب', 'غبي', 'حمار'}

IN  : وكلب
HITS: none



## 6. `_dist_stats(values)`
Computes distribution statistics (mean, std, min, max, percentiles) for any list of numbers.

In [44]:
import json

values = [10, 50, 200, 500, 1000, 1500, 3000, 150, 80, 400]
result = W._dist_stats(values)

print(f"IN : {values}")
print(f"OUT:")
for k, v in result.items():
    if k != "samples":
        print(f"  {k:<6}: {v}")

IN : [10, 50, 200, 500, 1000, 1500, 3000, 150, 80, 400]
OUT:
  n     : 10
  mean  : 689.0
  std   : 893.5485437288788
  min   : 10.0
  p25   : 97.5
  p50   : 300.0
  p75   : 875.0
  p95   : 2324.999999999998
  p99   : 2865.0
  max   : 3000.0
  sum   : 6890.0


In [45]:
# Edge case: empty list
result = W._dist_stats([])
print(f"IN : []")
print(f"OUT: {result}")

IN : []
OUT: {'n': 0}


In [46]:
# Single value
result = W._dist_stats([42.0])
print(f"IN : [42.0]")
print(f"OUT: mean={result['mean']}, min={result['min']}, max={result['max']}, std={result['std']}")

IN : [42.0]
OUT: mean=42.0, min=42.0, max=42.0, std=0.0


## 7. `stream_records(path)`
Reads a JSONL file line by line and yields each record as a Python dict. Skips blank lines and malformed JSON.

In [47]:
from pathlib import Path

path = Path("state_output_sample1000.jsonl")
records = list(W.stream_records(path))
print(f"Total records loaded: {len(records)}")

# Show the first record's fields
rec = records[0]
r = rec["record"]
print(f"\nFirst record:")
print(f"  status      : {rec['status']}")
print(f"  url         : {r.get('url', '')[:60]}")
print(f"  language    : {(r.get('langdetect') or {}).get('lang')}")
print(f"  lang score  : {(r.get('langdetect') or {}).get('score'):.3f}")
print(f"  timestamp   : {r.get('timestamp')}")
print(f"  content_type: {r.get('content_type')}")
print(f"  text preview: {(r.get('text') or '')[:80]}...")

Total records loaded: 1000

First record:
  status      : PASSED
  url         : http://2ahwasada.blogspot.com/2012/03/blog-post_6008.html
  language    : ar
  lang score  : 0.968
  timestamp   : 2017-08-22T14:44:09Z
  content_type: text/plain
  text preview: فنجان قهوة سادة
skip to main | skip to sidebar
فنجان قهوة سادة
ليس ‏هناك مذاق اس...


## 8. Full `analyze()` pipeline
Runs all analyses on the loaded records and returns a results dictionary.

In [48]:
from pathlib import Path

path = Path("state_output_sample1000.jsonl")
records = list(W.stream_records(path))
results = W.analyze(records)

print("=== CORPUS OVERVIEW ===")
t = results["totals"]
print(f"  Documents : {t['n_documents']:,}")
print(f"  Tokens    : {t['n_tokens']:,}")
print(f"  Bytes     : {t['n_bytes']:,}")
print(f"  Unique URLs: {t['n_unique_urls']:,}")
print(f"  Vocab size: {results['vocab']['n_types']:,}")

=== CORPUS OVERVIEW ===
  Documents : 1,000
  Tokens    : 1,028,842
  Bytes     : 10,097,974
  Unique URLs: 1,000
  Vocab size: 101,648


In [49]:
print("=== EXACT DUPLICATES ===")
ed = results["exact_duplicates"]
print(f"  Groups : {ed['n_dup_groups']}")
print(f"  Docs   : {ed['n_dup_docs']}")
for g in ed["top_groups"]:
    print(f"  count={g['count']}  url={g['url']}")
    print(f"  preview: {g['preview'][:80]!r}")
    print()

=== EXACT DUPLICATES ===
  Groups : 3
  Docs   : 10
  count=5  url=http://www.aljazeera.net/Services/sendtofriend/25867a71-a720-4f22-9f54-d72b1e7c9045
  preview: 'إرسال إلى صديق\nخطأ في إسم المستخدم أو كلمة السر الرجاء إدخال المعلومات بشكل صحيح'

  count=3  url=http://www.1808080.com/viewlisting.php?view=59196
  preview: '« اعلانات 1808080 كوم الكويت\nمرحبا بك أيها الضيف: دخول تسجيل RSS Feed\n1808080.co'

  count=2  url=http://www.jawhara-soft.com/vb/showthread.php?t=79309
  preview: 'منتدى التعليم التونسي (Jawhara-Soft)\nمنتديات جوهرة سوفت - Jawhara-Soft Forums\nمن'



In [50]:
print("=== NEAR DUPLICATES ===")
nd = results["near_duplicates"]
print(f"  Pairs    : {nd['n_pairs']}")
print(f"  Clusters : {nd['n_clusters']}")
print(f"  Threshold: {nd['threshold']}")
print()
for p in nd["example_pairs"][:3]:
    print(f"  A: {p['a_url']}")
    print(f"  B: {p['b_url']}")
    print(f"  A preview: {p['a_preview'][:60]!r}")
    print(f"  B preview: {p['b_preview'][:60]!r}")
    print()

=== NEAR DUPLICATES ===
  Pairs    : 109
  Clusters : 47
  Threshold: 0.8

  A: http://www.daralakhbar.com/videos/120497-%D8%A7%D9%84%D9%85%D9%82%D8%A7%D9%88%D9%85%D8%A9-%D8%A7%D9%84%D8%B4%D8%B9%D8%A8%D9%8A%D8%A9-%D8%AA%D8%B3%D8%AA%D8%B9%D9%8A%D8%AF-%D9%85%D9%86%D8%A7%D8%B7%D9%82-%D9%81%D9%8A-%D8%AA%D8%B9%D8%B2
  B: http://www.daralakhbar.com/videos/159384-%D9%84%D9%82%D8%A7%D8%A1%D8%A7%D8%AA-%D9%85%D8%AE%D8%AA%D9%84%D9%81%D8%A9-%D9%88%D8%AD%D8%B5%D8%B1%D9%8A%D8%A9-%D9%85%D8%B9-%D9%85%D9%85%D8%AB%D9%84%D9%8A%D9%86-%D8%AA%D9%88%D9%86%D8%B3%D9%8A%D9%8A%D9%86-%D9%88%D8%B9%D8%B1%D8%A8
  A preview: 'المقاومة الشعبية تستعيد مناطق في تعز دارالاخبار كوم\nدار الأخ'
  B preview: 'لقاءات مختلفة وحصرية مع ممثلين تونسيين وعرب في مهرجان قرط...'

  A: http://www.daralakhbar.com/videos/201114-%D9%81%D9%8A%D8%AF%D9%8A%D9%88-%D9%84%D8%AD%D8%B8%D8%A9-%D8%A7%D9%86%D9%81%D8%AC%D8%A7%D8%B1-%D9%82%D8%A7%D9%85-%D8%A8%D9%87-%D8%A3%D8%AD%D8%AF-%D9%85%D9%86%D9%81%D8%B0%D9%8A-%D9%87%D8%AC%D9%85%D8%A7%D8%AA-%D8%A8

In [51]:
print("=== TOP UNIGRAMS (stopwords removed) ===")
for word, count in results["top_unigrams"][:10]:
    print(f"  {word}: {count}")

=== TOP UNIGRAMS (stopwords removed) ===
  2017: 3935
  منتدي: 3157
  الله: 2483
  10: 2330
  اخبار: 2129
  08: 2116
  طرف: 2000
  12: 1928
  المنتدي: 1847
  11: 1839


In [52]:
print("=== PII DETECTION ===")
pii = results["pii"]
for cat in ("email", "ipv4", "phone", "url", "card_like"):
    print(f"  {cat:<12}: {pii['counts'][cat]:>4} matches in {pii['docs_containing'][cat]:>3} docs")
    if pii["examples"][cat]:
        print(f"    examples: {pii['examples'][cat][:3]}")

=== PII DETECTION ===
  email       :   76 matches in  51 docs
    examples: ['ahmed_club2000@yahoo.com', 'info@kds.ae', 'pro-coder@nimbuzz.com']
  ipv4        :   17 matches in  13 docs
    examples: ['217.218.48.21', '18.0.0.405', '18.0.0.405']
  phone       : 3819 matches in 500 docs
    examples: ['2017 - 29', '01282708137', '1 2 3 4 5 6 7 8']
  url         :  599 matches in 146 docs
    examples: ['http://www.all-in-one.own0.com', 'http://rapidshare.com/files/245777386/Ydontmzohan.part1.rar', 'https://youtu.be/VqU6H0BJMuk']
  card_like   :  939 matches in 117 docs
    examples: ['1515 1516 1517 1518 ', '3297 3298 3299 3300 ', '13 14 15 16 17 18 19']


In [53]:
print("=== OFFENSIVE CONTENT ===")
off = results["offensive"]
print(f"  Documents with offensive content: {off['n_docs_with_offensive']}")
print(f"  Top terms: {off['top_terms'][:5]}")

=== OFFENSIVE CONTENT ===
  Documents with offensive content: 30
  Top terms: [('كلب', 14), ('لعنه', 7), ('غبي', 6), ('كافر', 5), ('وسخ', 5)]


In [54]:
print("=== SELF CONTAMINATION ===")
sc = results["self_contamination"]
print(f"  N-gram size       : {sc['ngram_size']}")
print(f"  Repeated n-grams  : {sc['n_repeated_long_ngrams']:,}")
print()
for ex in sc["examples"][:2]:
    print(f"  doc A: {ex['doc_a_url']}")
    print(f"  doc B: {ex['doc_b_url']}")
    print(f"  span : {ex['gram_preview'][:80]!r}")
    print()

=== SELF CONTAMINATION ===
  N-gram size       : 50
  Repeated n-grams  : 70,778

  doc A: http://arabwikileaks.alafdal.net/t440-topic
  doc B: http://arabwikileaks.alafdal.net/t693-topic
  span : 'arabwikileaks اهلا وسهلا بك عزيزي الزائر قم بالتسجيل حتي تتمكن من رؤيه الموضوع و'

  doc A: http://arabwikileaks.alafdal.net/t440-topic
  doc B: http://arabwikileaks.alafdal.net/t693-topic
  span : 'اهلا وسهلا بك عزيزي الزائر قم بالتسجيل حتي تتمكن من رؤيه الموضوع ونرجوا منك ترك '



## 9. N-gram analysis 
The WIMBD paper highlights n-gram analysis as a key data quality signal — the most common n-grams often reveal boilerplate, noise, or template-generated content. Here we run the full normalize → tokenize → stopword-remove → count pipeline on a real Arabic sentence, then compare with corpus-level results.

In [55]:
# Single-sentence n-gram demo — mirrors what analyze() does internally
import collections

text = "أنـــا سعيدٌ جداً بوجودي في هذه الوَرْشَةِ مع United Nations عام 2022"

# Step 1: normalize
normalized = W.normalize_ar(text)
print(f"IN        : {text}")
print(f"normalized: {normalized}")
print()

# Step 2: tokenize
tokens = W.tokenize(normalized)
print(f"tokens    : {tokens}")
print()

# Step 3: remove stopwords (len > 1)
nostop = [t for t in tokens if t not in W.ARABIC_STOPWORDS and len(t) > 1]
print(f"no stops  : {nostop}")
print()

# Step 4: compute unigrams, bigrams, trigrams
uni = collections.Counter(nostop)
bi  = collections.Counter(zip(nostop, nostop[1:]))
tri = collections.Counter(zip(nostop, nostop[1:], nostop[2:]))

print(f"unigrams : {list(uni.most_common())}")
print(f"bigrams  : {[(' '.join(g), c) for g, c in bi.most_common()]}")
print(f"trigrams : {[(' '.join(g), c) for g, c in tri.most_common()]}")

IN        : أنـــا سعيدٌ جداً بوجودي في هذه الوَرْشَةِ مع United Nations عام 2022
normalized: انا سعيد جدا بوجودي في هذه الورشه مع United Nations عام 2022

tokens    : ['انا', 'سعيد', 'جدا', 'بوجودي', 'في', 'هذه', 'الورشه', 'مع', 'United', 'Nations', 'عام', '2022']

no stops  : ['سعيد', 'بوجودي', 'الورشه', 'United', 'Nations', 'عام', '2022']

unigrams : [('سعيد', 1), ('بوجودي', 1), ('الورشه', 1), ('United', 1), ('Nations', 1), ('عام', 1), ('2022', 1)]
bigrams  : [('سعيد بوجودي', 1), ('بوجودي الورشه', 1), ('الورشه United', 1), ('United Nations', 1), ('Nations عام', 1), ('عام 2022', 1)]
trigrams : [('سعيد بوجودي الورشه', 1), ('بوجودي الورشه United', 1), ('الورشه United Nations', 1), ('United Nations عام', 1), ('Nations عام 2022', 1)]


In [56]:
# Boilerplate detection demo — the WIMBD paper found repeated punctuation and
# template text dominate top n-grams in low-quality corpora.
# Here we show what happens with a forum boilerplate string like those found
# in our corpus (arabwikileaks.alafdal.net).

boilerplate = "اهلا وسهلا بك عزيزي الزائر قم بالتسجيل حتي تتمكن من رؤيه الموضوع ونرجوا منك ترك تعليق"

norm = W.normalize_ar(boilerplate)
toks = W.tokenize(norm)
nostop = [t for t in toks if t not in W.ARABIC_STOPWORDS and len(t) > 1]

print(f"IN      : {boilerplate}")
print(f"tokens  : {toks}")
print(f"no stops: {nostop}")
print()
print("If this text appears in thousands of documents, these tokens will")

IN      : اهلا وسهلا بك عزيزي الزائر قم بالتسجيل حتي تتمكن من رؤيه الموضوع ونرجوا منك ترك تعليق
tokens  : ['اهلا', 'وسهلا', 'بك', 'عزيزي', 'الزائر', 'قم', 'بالتسجيل', 'حتي', 'تتمكن', 'من', 'رؤيه', 'الموضوع', 'ونرجوا', 'منك', 'ترك', 'تعليق']
no stops: ['اهلا', 'وسهلا', 'عزيزي', 'الزائر', 'قم', 'بالتسجيل', 'تتمكن', 'رؤيه', 'الموضوع', 'ونرجوا', 'ترك', 'تعليق']

If this text appears in thousands of documents, these tokens will


In [57]:
# Corpus-level n-gram results from the 1K sample
# (requires results dict from Section 8 above)
print("TOP UNIGRAMS (stopwords removed) — WIMBD §4.3.1")
print("-" * 40)
for word, count in results["top_unigrams"][:15]:
    bar = '█' * int(count / 200)
    print(f"  {word:<15} {count:>5}  {bar}")

print()
print("TOP BIGRAMS")
print("-" * 40)
for bigram, count in results["top_bigrams"][:10]:
    print(f"  {bigram:<25} {count:>5}")

print()
print("TOP TRIGRAMS")
print("-" * 40)
for trigram, count in results["top_trigrams"][:10]:
    print(f"  {trigram:<35} {count:>5}")

print()
print("Stopword normalization fix applied:")
print("  ARABIC_STOPWORDS = {normalize_ar(w) for w in _ar_sw.stopwords_list()}")
print("  'ان', 'الي', 'او', 'هن', 'والى' no longer appear in top unigrams after fix.")
print()
print("Remaining noise in unigrams:")
print("  '2017', '10', '08' = timestamps/numbers leaking from metadata.")
print("  'Co Ltd مقاطعه', 'Guangdong China' in trigrams = Chinese product listings contamination.")

TOP UNIGRAMS (stopwords removed) — WIMBD §4.3.1
----------------------------------------
  2017             3935  ███████████████████
  منتدي            3157  ███████████████
  الله             2483  ████████████
  10               2330  ███████████
  اخبار            2129  ██████████
  08               2116  ██████████
  طرف              2000  ██████████
  12               1928  █████████
  المنتدي          1847  █████████
  11               1839  █████████
  قسم              1733  ████████
  22               1716  ████████
  2016             1671  ████████
  يا               1549  ███████
  اغسطس            1514  ███████

TOP BIGRAMS
----------------------------------------
  pm طرف                     1043
  تاريخ التسجيل               908
  اغسطس 2017                  735
  نموذج رقم                   659
  Co Ltd                      631
  Ltd مقاطعه                  608
  سعر الوحده                  592
  الوحده US                   592
  am طرف                      560
  22 اغسط